# SECOM Data Modeling
---

### Imports and creating test-train split

In [42]:
# Imports
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier
from sklearn.cross_decomposition import PLSRegression

In [43]:
# Pulling in the SECOM data and loading data into feature and label dataframes
secom = fetch_ucirepo(id=179)
df = pd.DataFrame(secom.data.original)
X = df.drop(columns=["class", "timestamp"])
y = df["class"]
# converting "-1" passing label to "0"
y = y.replace(-1, 0)

In [44]:
# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [45]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

class
0    0.933759
1    0.066241
Name: proportion, dtype: float64
class
0    0.933121
1    0.066879
Name: proportion, dtype: float64


### Creating custom Secom dataset preprocess transformer

In [46]:
class SecomPreProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing_threshold: int, corr_threshold: int):
        self.missing_threshold = missing_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X: pd.DataFrame, y=None):
        X = X.copy()

        # filtering out features with null percentage above threshold
        self.null_cols_ = X.columns[X.isna().mean() > self.missing_threshold].to_list()
        X = X.drop(columns=self.null_cols_)
        
        # imputing NaN with median
        self.medians_ = X.median()
        X = X.fillna(self.medians_)

        # Remove constant variance features
        self.zero_var_cols_ = X.columns[X.var() == 0].to_list()
        X = X.drop(columns=self.zero_var_cols_)

        # dropping columns correlated above threshold
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        self.corr_cols_ = [
            col
            for col in upper.columns
            if any(upper[col] > self.corr_threshold)
        ]

        X = X.drop(columns=self.corr_cols_)

        self.feature_names_out_ = X.columns.to_list()

        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        X = X.reindex(columns=self.feature_names_out_)

        X = X.fillna(self.medians_)

        return X

    def get_feature_names_out(self, input_features=None) -> np.array:
        return np.array(self.feature_names_out_)

In [47]:
# Creating custom transformer to extract PLS componenets
class PLSTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=10):
        self.n_components = n_components

    def fit(self, X, y):
        self.pls_ = PLSRegression(n_components=self.n_components)
        self.pls_.fit(X, y)
        return self

    def transform(self, X):
        return self.pls_.transform(X)

___

### Creating a loop to test the performance of several models at once

In [48]:
models_needs_scaling = {
    "Logistic Regression": LogisticRegression(
        penalty='l1', 
        random_state=42, 
        class_weight="balanced", 
        solver='liblinear', 
        max_iter=5000        
    )
}
models_no_scaling = {
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Random Forest (small)": RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10,
        min_samples_split=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Histogram Gradient Boost": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=4,
        max_iter=300,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42,
        eval_metric="logloss"
    )
}

In [49]:
models_custom_pipelines = {
    f"PLS-DA {n} components": Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("scale", StandardScaler()),
        ("pls", PLSTransformer(n_components=n)),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    ])
    for n in [2, 5, 10, 15, 20, 30]
}

In [50]:
def run_performance(models_needs_scaling: dict, models_no_scaling: dict, models_custom_pipelines: dict | None = None, 
                    X_train : pd.DataFrame = X_train, y_train: pd.DataFrame = y_train) -> pd.DataFrame:
    results = []

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def evaluate_pipe(name, pipe):
        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "pr_auc": "average_precision"
            },
            return_train_score=True,
            n_jobs=-1
        )

        results.append({
            "model": name,
            "train_roc_auc": scores["train_roc_auc"].mean(),
            "train_pr_auc": scores["train_pr_auc"].mean(),
            "test_roc_auc_mean": scores["test_roc_auc"].mean(),
            "test_roc_auc_std": scores["test_roc_auc"].std(),
            "test_pr_auc_mean": scores["test_pr_auc"].mean(),
            "test_pr_auc_std": scores["test_pr_auc"].std(),
        })


    for name, model in models_needs_scaling.items():
        pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("scale", StandardScaler()),
        ("model", model)
        ])

        evaluate_pipe(name, pipe)
        

    for name, model in models_no_scaling.items():
        pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("model", model)
        ])

        evaluate_pipe(name, pipe)

    if models_custom_pipelines is not None:
        for name, pipe in models_custom_pipelines.items():
            evaluate_pipe(name, pipe)

    performance_summary = pd.DataFrame(results).sort_values("test_pr_auc_mean", ascending=False).reset_index(drop=True)
    return performance_summary

In [51]:
performance_summary = run_performance(models_needs_scaling=models_needs_scaling, models_no_scaling=models_no_scaling, 
                                      models_custom_pipelines=models_custom_pipelines)
performance_summary

,model,train_roc_auc,train_pr_auc,test_roc_auc_mean,test_roc_auc_std,test_pr_auc_mean,test_pr_auc_std
0,Random Forest (small),1.000000,1.000000,0.724020,0.048940,0.200041,0.052828
1,Random Forest,1.000000,1.000000,0.722766,0.054694,0.198035,0.059456
2,XGBoost,1.000000,1.000000,0.698303,0.063096,0.194247,0.048157
3,PLS-DA 2 components,0.901980,0.529548,0.662689,0.058279,0.161826,0.057569
4,PLS-DA 5 components,0.944674,0.702331,0.646732,0.037519,0.153672,0.052328
5,Logistic Regression,0.998701,0.970597,0.599604,0.082212,0.153012,0.079866
6,PLS-DA 20 components,0.980379,0.836153,0.646952,0.045097,0.143164,0.036943
7,Histogram Gradient Boost,1.000000,1.000000,0.655505,0.061309,0.142040,0.017916
8,PLS-DA 30 components,0.983088,0.833634,0.636105,0.046496,0.136561,0.039326
9,PLS-DA 15 components,0.974985,0.821324,0.651882,0.046868,0.134008,0.032091


In [10]:
best_pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10,
        min_samples_split=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
        ))
])

In [11]:
def threshold_sweep(best_pipe: Pipeline, thresholds: list[int], X_train: pd.DataFrame = X_train, 
                    y_train: pd.DataFrame = y_train, X_test: pd.DataFrame = X_test, y_test: pd.DataFrame = y_test) -> pd.DataFrame:
    best_pipe.fit(X_train, y_train)
    y_proba = best_pipe.predict_proba(X_test)[:,1]

    baseline_failure_rate = y_test.mean()

    results = []

    for threshold in thresholds:
        y_pred = (y_proba > threshold).astype(int)

        precision = precision_score(y_test, y_pred, zero_division=0)

        results.append({
            "threshold": threshold,
            "precision": precision,
            "recall": recall_score(y_test, y_pred),
            "f1" : f1_score(y_test, y_pred),
            "flagged": y_pred.sum(),
            "enrichment": round(precision/baseline_failure_rate, 2)
        })
    threshold_df = pd.DataFrame(results).sort_values("threshold", ascending=False).reset_index(drop=True)
    return threshold_df

In [12]:
thresholds = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
threshold_df = threshold_sweep(best_pipe=best_pipe, thresholds=thresholds)
threshold_df

,threshold,precision,recall,f1,flagged,enrichment
0,0.50,0.250000,0.142857,0.181818,12,3.74
1,0.40,0.236842,0.428571,0.305085,38,3.54
2,0.30,0.142857,0.714286,0.238095,105,2.14
3,0.20,0.085470,0.952381,0.156863,234,1.28
4,0.10,0.067093,1.000000,0.125749,313,1.00
5,0.05,0.066879,1.000000,0.125373,314,1.00


In [ ]:
feature_importances = best_pipe.named_steps["model"].feature_importances_
feature_names = best_pipe.named_steps["preprocess"].get_feature_names_out()

feature_importance_df = pd.DataFrame({
    "sensor": feature_names,
    "feature_importance": feature_importances
    })

feature_importance_df.sort_values("feature_importance", ascending=False).head(20)

In [ ]:
# perm = permutation_importance(
#     best_pipe,
#     X_test,
#     y_test,
#     scoring="average_precision",
#     n_repeats=20,
#     random_state=42,
#     n_jobs=-1
# )

In [ ]:
# perm_df = pd.DataFrame({
#     "feature": X_test.columns,
#     "importance_mean": perm.importances_mean,
#     "importance_std": perm.importances_std
# }).sort_values("importance_mean", ascending=False)

# perm_df.head(20)